# CSE 4095 — Class 4: Comparing Models Fairly
Use AI to help design and implement the workflow, but you must control the comparison.

## 1. Choose your track
Choose classification (`high_exam_score`) or regression (`exam_score`). Write your prediction scenario.

In [ ]:
# TODO: load dataset and choose classification or regression track

try:
    from google.colab import files
    uploaded = files.upload()
except ImportError:
    print("Not running in Colab. Using class1_student_habits.csv from this folder.")

import pandas as pd
import matplotlib.pyplot as plt

# "None" is a real answer for prior programming experience.
# The default reader would treat that word as a missing value.
df = pd.read_csv(
    "class1_student_habits.csv",
    keep_default_na=False,
    na_values=[""],
)

df["high_exam_score"] = "No"
df.loc[df["exam_score"] >= 80, "high_exam_score"] = "Yes"

print(df[["student_id", "exam_score", "high_exam_score"]].head(10))
print()

class_counts = df["high_exam_score"].value_counts()
class_percentages = df["high_exam_score"].value_counts(normalize=True) * 100

print("Number of students:", len(df))
print()
print("Students in each class:")
print(class_counts)
print()
print("Percentage of students in each class:")
print(class_percentages.round(1))

order = ["No", "Yes"]
counts = df["high_exam_score"].value_counts().reindex(order)

plt.figure(figsize=(6, 4))
plt.bar(order, counts, edgecolor="black")
plt.title("Class Distribution of high_exam_score")
plt.xlabel("high_exam_score")
plt.ylabel("Number of students")
plt.show()

## Part 1 Questions
1. What are you trying to predict? Whether a student will earn a high exam score. high_exam_score is Yes when exam_score is 80 or higher and No when it is below 80. The goal is to predict that label from study habits and background, before the exam, rather than to estimate the points themselves.

2. Why is this a classification problem? The target has two categories, Yes and No. The model is asked which group a student belongs to. A prediction is either the right label or the wrong label. A score of 81 and a score of 100 are both Yes. A score of 79 and a score of 51.4 are both No. How far the score sits from 80 does not change the class.

3. What percentage of students belong to each class? 39.6% are Yes and 60.4% are No.

4. Is the dataset approximately balanced? Yes, approximately. Neither class is rare. No is the larger class, about 60% of the students, and Yes is about 40%. A majority-class guess of No would be right about 60% of the time, so later accuracy numbers need to be compared with that baseline. The split is uneven enough to notice, and both classes are well represented.

## 2. Define X and y
Use legitimate features. Explain excluded columns.

In [ ]:
# TODO: define X, y, and feature columns

feature_columns = [
    "study_hours_per_week",
    "sleep_hours_per_night",
    "attendance_rate",
    "practice_problems_per_week",
    "prior_programming_experience",
    "study_group",
    "commute_minutes",
]

target_column = "high_exam_score"

X = df[feature_columns]
y = df[target_column]

print("Predictors:")
print(list(X.columns))
print()
print("Target:", target_column)
print()
print("Left out of the model:")
print(["student_id", "exam_score", "quiz_average"])
print()
print("X shape:", X.shape)
print("y shape:", y.shape)

X.head()

## Part 2 Question

Why would using exam_score be target leakage? high_exam_score was built directly from exam_score, since Yes if the score is 80 or higher, No otherwise. If the model can see the exam score, it does not need study hours, sleep, or attendance. It can apply the same cutoff and get the label right every time. That accuracy would be meaningless, because the feature is the answer written in another form. This is target leakage. For a student who has not taken the exam yet, the score would not be available, which is why the habits are the predictors and the exam score is not.

## 3. Train/test split
Create an 80/20 split. Keep the test set for final evaluation.

In [ ]:
# TODO: train/test split

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

def class_mix(labels):
    counts = labels.value_counts()
    percentages = labels.value_counts(normalize=True) * 100
    summary = pd.DataFrame({"count": counts, "percent": percentages.round(1)})
    return summary.reindex(["No", "Yes"])

print("Training rows:", len(X_train))
print(class_mix(y_train))
print()
print("Test rows:", len(X_test))
print(class_mix(y_test))
print()
print("The test set is held out. It is not used again until the final evaluation.")

## Part 3 Questions

1. Why are we holding the test set aside? The test set is the check on students the model has not learned from. There are 50 of them, 20% of the class. Model comparison and hyperparameter choices are made from the 200 training students, using cross-validation. If we look at the test students while we are still choosing, the test set stops being an honest check of how the model would do on someone new. We use it once, after the model is already chosen.

2. What could happen if we repeatedly check test accuracy while modifying our model? Each change we keep because it scored higher on the test set is a choice fitted to those 50 students. The test accuracy can rise because we picked what worked on that particular group, not because the model learned a pattern that will hold for a new student. The final number then looks better than the model really is.

## 5. Preprocessing pipeline
Handle missing values and categorical variables.

In [ ]:
# TODO: preprocessing setup

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features = [
    "study_hours_per_week",
    "sleep_hours_per_night",
    "attendance_rate",
    "practice_problems_per_week",
    "commute_minutes",
]
categorical_features = [
    "prior_programming_experience",
    "study_group",
]

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

print("Numeric features: median impute, then StandardScaler")
print(numeric_features)
print()
print("Categorical features: most-frequent impute, then one-hot encoding")
print(categorical_features)
print()
print("Missing values in the training set:")
print(X_train[numeric_features + categorical_features].isna().sum())


## Part 4 Questions

1. Why should preprocessing be included inside the pipeline? The imputer, the scaler, and the encoder all learn from data. The numeric branch records each column's median, then its mean and standard deviation. The categorical branch records the most common category and the category names that become 0/1 columns. If those steps sit outside the model, it is easy to learn them from rows the model is not supposed to see. Inside a pipeline, fit learns those values from the rows passed to it, and later rows are prepared with those same values. The preparation and the model stay one workflow.

2. Why is this especially important when using cross-validation? Each fold holds out a different slice of the training data as validation. If we impute and scale the full training set first, then run cross-validation, the validation rows help set the medians, means, and standard deviations used on that fold's training rows. The validation score then looks a little better than a real new student would get, because information from the held-out fold leaked into the preparation. With the preprocessor inside the pipeline, each fold refits the imputer and scaler on that fold's training rows only, then applies them to that fold's validation rows.

3. Which of the models in this assignment are sensitive to feature scaling? Logistic Regression. It multiplies each feature by a weight, and the default model also penalizes large weights. Study hours are small numbers, attendance is usually in the 70s to 100, and commute minutes can be near 0 or well over 100. Without scaling, a column with bigger numbers can push the model harder just because of its units. After StandardScaler, a value of 1 means one standard deviation above that column's training average, so the weights are comparable.

4. Do Decision Trees and Random Forests generally require scaling? No. A tree asks questions such as whether study hours are greater than 6. The split depends on the order of values in that one column. Changing the units moves the threshold, and the groups stay the same. A random forest is a collection of those trees, so it does not need StandardScaler either. The shared pipeline still scales every model, because Logistic Regression is in the comparison and all three models should see the same prepared columns.

## 4. Baseline
Create a majority-class baseline for classification or mean baseline for regression.

In [ ]:
# TODO: baseline

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

baseline = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("classifier", DummyClassifier(strategy="most_frequent")),
])

baseline_scores = cross_val_score(
    baseline, X_train, y_train, cv=cv, scoring="accuracy"
)

print("Most common training class:", y_train.value_counts().idxmax())
print("Training class counts:")
print(y_train.value_counts())
print()
print("Fold accuracies:", [round(score, 3) for score in baseline_scores])
print("Mean CV accuracy:", round(baseline_scores.mean(), 3))
print("Standard deviation:", round(baseline_scores.std(), 3))

## Part 5 Questions

1. What does the baseline do? It ignores study hours, sleep, attendance, and the other features. On each fold it finds the most common class in that fold's training rows and predicts that class for every student in the held-out rows. In this training set the most common class is No: 121 of the 200 students scored below 80. So the baseline always predicts No.

2. What is its mean CV accuracy? 0.605, or 60.5%. The five fold accuracies are 0.625, 0.600, 0.600, 0.600, and 0.600. The standard deviation across folds is 0.010. One validation fold has 25 No students out of 40, and the other four have 24 out of 40. Because the prediction is always No, those rates are the accuracies.

3. Why should our machine-learning models be compared with this baseline? A model is only doing useful work if it beats always saying No. An accuracy around 60% can look acceptable on its own and still mean the model learned nothing about the study habits. Later models should be judged against this 60.5% cross-validation accuracy, using the same training students and the same five folds.

## 6. Cross-validation comparison
Compare a small set of models using the same folds and metrics.

In [ ]:
# TODO: cross-validation model comparison

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

print("Training students:", len(X_train))
print("Each fold validates on", len(X_train) // 5, "students and trains on", len(X_train) - len(X_train) // 5)
print("The test set is not used.")
print()

models = {
    "Baseline": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
}

cv_results = {}

for name, model in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", clone(preprocessor)),
        ("classifier", model),
    ])
    scores = cross_val_score(
        pipeline, X_train, y_train, cv=cv, scoring="accuracy"
    )
    cv_results[name] = scores
    print(name)
    print("  Fold accuracies:", [round(score, 3) for score in scores])
    print("  Mean CV accuracy:", round(scores.mean(), 3))
    print("  Standard deviation:", round(scores.std(), 3))
    print()

## Part 6 Questions

1. What happens during 5-fold cross-validation? The 200 training students are divided into five groups of 40. The model trains on four groups, 160 students, and is scored on the remaining group. Then the held-out group changes. In fold 1 the first group is the validation set. In fold 2 the second group is, and so on, until every student has been in the validation set once. Each fold refits the imputer, the scaler, and the encoder on that fold's training rows only. The five accuracies are averaged, and the standard deviation shows how much the score moves when the held-out students change. The 50 test students are not part of this.

2. How many times is each model trained? Five times. Once for each fold, on the four groups that are not being validated. Logistic Regression, the decision tree, and the random forest are each fit five times on the training data. The cross-validation score is the average of those five checks, not a sixth fit.

3. Why is cross-validation more informative than evaluating a model using only one training/validation split? One split of 160 and 40 can be lucky or unlucky. The validation students might be easy to classify, or unusually hard, and a single accuracy would hide that. Five folds give five scores, and every training student is validated once. The mean is a steadier estimate, and the spread shows whether the result depends on which 40 students were held out. Logistic Regression's folds, for example, run from 0.65 to 0.85. A single split could have landed on either end and told a different story.

4. Why do we still keep a separate final test set? The cross-validation scores are used to compare models, so those 200 students play a role in the choice. After we have looked at their validation scores, they are no longer a fully fresh check. The 50 test students were set aside before any comparison. They are used once, after a model is chosen, to estimate how it does on students that took no part in the decision.

## Part 7 — Compare Three Classification Models

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

logistic_model = LogisticRegression(max_iter=1000, random_state=42)
tree_model = DecisionTreeClassifier(max_depth=4, random_state=42)
forest_model = RandomForestClassifier(n_estimators=200, random_state=42)

classifiers = {
    "Majority baseline": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": logistic_model,
    "Decision Tree": tree_model,
    "Random Forest": forest_model,
}

print("Logistic Regression")
print(logistic_model)
print()
print("Decision Tree")
print(tree_model)
print()
print("Random Forest")
print(forest_model)


## Part 8 — Compare the Models with Cross-Validation

In [ ]:
from sklearn.model_selection import cross_validate
from sklearn.metrics import make_scorer, precision_score, recall_score

scoring = {
    "accuracy": "accuracy",
    "precision": make_scorer(precision_score, pos_label="Yes", zero_division=0),
    "recall": make_scorer(recall_score, pos_label="Yes", zero_division=0),
}

rows = []

for name, model in classifiers.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", clone(preprocessor)),
        ("classifier", model),
    ])
    scores = cross_validate(
        pipeline, X_train, y_train, cv=cv, scoring=scoring
    )
    rows.append({
        "Model": name,
        "Mean CV Accuracy": scores["test_accuracy"].mean(),
        "Std. Accuracy": scores["test_accuracy"].std(),
        "Precision": scores["test_precision"].mean(),
        "Recall": scores["test_recall"].mean(),
    })
    print(name)
    print("  Accuracy folds:", [round(score, 3) for score in scores["test_accuracy"]])
    print("  Precision folds:", [round(score, 3) for score in scores["test_precision"]])
    print("  Recall folds:", [round(score, 3) for score in scores["test_recall"]])
    print()

comparison_table = pd.DataFrame(rows).round(3)
comparison_table

## Part 8 Questions

1. Which model has the highest mean CV accuracy? Logistic Regression, at 0.740. 

2. How large are the differences? Logistic regression is 0.050 above the random forest and 0.070 above the decision tree. The forest is 0.085 above the baseline, and the tree is 0.065 above it. Logistic regression is 0.135 above always predicting No. The 0.050 gap between the two best models is smaller than logistic regression's own fold-to-fold standard deviation of 0.064, so the average ranking is clear and one unlucky fold could still put the forest ahead.

3. Which model has the highest recall? Logistic Regression, with a mean recall of 0.571 for the Yes class. The random forest is 0.518 and the decision tree is 0.493. The baseline recall is 0. The baseline never predicts Yes, so it never catches a student who scores 80 or above.

4. Do all metrics identify the same model? Yes. Logistic regression is highest on mean accuracy (0.740), precision (0.728), and recall (0.571). The order is the same on every column: logistic regression, then random forest, then the decision tree, then the baseline. Accuracy still makes the baseline look closer than it is. Always predicting No is right about 60.5% of the time, while its precision and recall for Yes are both 0.

5. How much does performance vary between folds? The baseline barely moves, from 0.600 to 0.625, with a standard deviation of 0.010. Logistic regression's accuracy runs from 0.650 to 0.850, standard deviation 0.064. The decision tree runs from 0.625 to 0.725, standard deviation 0.040. The random forest runs from 0.625 to 0.800, standard deviation 0.072. Recall moves more than accuracy. Logistic regression's Yes recall runs from 0.375 to 0.688, and the decision tree's runs from 0.375 to 0.750. The mean across five folds is the fairer comparison.

## 7. Tune one hyperparameter
Tune decision-tree max_depth using cross-validation on the training data.

In [ ]:
# TODO: tune max_depth

depths = [1, 2, 3, 4, 5, 6, 8, 10, None]
depth_rows = []

for depth in depths:
    pipeline = Pipeline(steps=[
        ("preprocessor", clone(preprocessor)),
        ("classifier", DecisionTreeClassifier(max_depth=depth, random_state=42)),
    ])
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring="accuracy",
        return_train_score=True,
    )
    depth_rows.append({
        "max_depth": "None" if depth is None else depth,
        "Mean train accuracy": scores["train_score"].mean(),
        "Mean CV accuracy": scores["test_score"].mean(),
        "Std. CV accuracy": scores["test_score"].std(),
    })

best_index = max(range(len(depth_rows)), key=lambda i: depth_rows[i]["Mean CV accuracy"])
best_max_depth = depths[best_index]

depth_table = pd.DataFrame(depth_rows).round(3)
print(depth_table.to_string(index=False))
print()
print("Chosen max_depth:", best_max_depth)

plot_depths = ["None" if depth is None else str(depth) for depth in depths]
plt.figure(figsize=(8, 4))
plt.plot(plot_depths, depth_table["Mean train accuracy"], marker="o", label="Training accuracy")
plt.plot(plot_depths, depth_table["Mean CV accuracy"], marker="o", label="Validation accuracy")
plt.axhline(baseline_scores.mean(), linestyle="--", color="gray", label="Baseline")
plt.xlabel("max_depth")
plt.ylabel("Accuracy")
plt.title("Decision Tree Depth: Training vs Validation Accuracy")
plt.legend()
plt.show()

## Part 9 Question

A model parameter is learned from the data during training. When fit runs, the algorithm looks at the training rows and sets those values. A hyperparameter is chosen before training. It is a setting of the algorithm, and the training rows do not estimate it. For this decision tree, the parameters are the splits the tree settles on, which feature to test, the cutoff on that feature, and the class label stored in each leaf. Those come from the 160 students in a training fold. max_depth is a hyperparameter. We choose it before fit runs, and it limits how many layers of splits the tree may use. At depth 1 the tree is a single question. With no depth limit, training accuracy on these folds reaches 1.000, so the tree has separated the training students perfectly, while validation accuracy falls from 0.670 at depth 4 to 0.595. The extra layers are memorizing the training fold. Cross-validation on the training data picks depth 4, because that depth has the highest mean validation accuracy. The test set is not used for this choice.

## Part 10 — Decision-Tree Depth

Training accuracy is the mean accuracy on the rows each fold trains on. CV accuracy is the mean accuracy on the held-out fold. CV Std. is the standard deviation of those five validation accuracies. Unlimited depth is max_depth=None. 

In [ ]:
depth_experiment = [1, 2, 3, 4, 6, 8, None]
depth_experiment_rows = []

for depth in depth_experiment:
    pipeline = Pipeline(steps=[
        ("preprocessor", clone(preprocessor)),
        ("classifier", DecisionTreeClassifier(max_depth=depth, random_state=42)),
    ])
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring="accuracy",
        return_train_score=True,
    )
    depth_experiment_rows.append({
        "max_depth": "None" if depth is None else depth,
        "Training Accuracy": scores["train_score"].mean(),
        "CV Accuracy": scores["test_score"].mean(),
        "CV Std.": scores["test_score"].std(),
    })

depth_experiment_table = pd.DataFrame(depth_experiment_rows).round(3)
depth_experiment_table

## Part 11 — Visualize Model Complexity

The plot uses the Part 10 results. The x-axis is max_depth. The y-axis is accuracy. One line is the mean training accuracy. The other is the mean cross-validation accuracy.

In [ ]:
plot_labels = depth_experiment_table["max_depth"].astype(str)

plt.figure(figsize=(8, 4))
plt.plot(
    plot_labels,
    depth_experiment_table["Training Accuracy"],
    marker="o",
    label="Training accuracy",
)
plt.plot(
    plot_labels,
    depth_experiment_table["CV Accuracy"],
    marker="o",
    label="Cross-validation accuracy",
)
plt.xlabel("max_depth")
plt.ylabel("Accuracy")
plt.title("Decision Tree Complexity: Training vs Cross-Validation Accuracy")
plt.legend()
plt.show()

## Part 11 Questions

1. What happens to training accuracy as max_depth increases? It rises at every step. It is 0.661 at depth 1, 0.712 at depth 2, 0.772 at depth 3, 0.826 at depth 4, 0.925 at depth 6, and 0.979 at depth 8. With no depth limit it reaches 1.000. A deeper tree fits the training students more closely, until an unlimited tree separates them perfectly.

2. What happens to cross-validation accuracy? It does not keep rising with the training line. It moves from 0.560 at depth 1 to 0.625 at depth 2 and 0.610 at depth 3, then peaks at 0.670 at depth 4. After that it falls to 0.615 at depth 6, 0.620 at depth 8, and 0.595 with no limit. Deeper trees do worse on the held-out students once the tree passes depth 4.

3. At what depth does the tree appear to perform best on unseen data? Depth 4. Its mean cross-validation accuracy is 0.670, the highest of the seven depths, and its standard deviation is the smallest, 0.040.

4. Do you see evidence of underfitting? Yes, at depth 1. Training accuracy is only 0.661, and cross-validation accuracy is 0.560, which is below the majority baseline of 0.605. The tree is allowed one split, so it cannot capture the pattern in the study habits. Both lines are low, which is what underfitting looks like.

5. Do you see evidence of overfitting? Yes, after depth 4. Training accuracy keeps climbing toward 1.000 while cross-validation accuracy drops from 0.670 to 0.595. The gap between the two lines gets wider. With no depth limit the tree is perfect on the training rows and only 0.595 on the held-out rows, just under the baseline. The extra depth is memorizing the training fold.

## Part 12 — Underfitting and Overfitting

The Part 10 table and the Part 11 plot show three different behaviors.

Underfitting: the model is too simple to capture the pattern. Both training performance and validation performance are poor. Depth 1 is that case. Training accuracy is 0.661 and cross-validation accuracy is 0.560, which is below the majority baseline of 0.605. One split is not enough to use the study habits.

Good generalization: the model captures a pattern that also shows up on unseen students. Training performance is good and validation performance is good. Depth 4 is that case. Training accuracy is 0.826 and cross-validation accuracy is 0.670, the highest of the seven depths and well above the baseline.

Overfitting: the model fits the training rows extremely well and does worse on students it did not train on. Depth 8 and unlimited depth are that case. Training accuracy is 0.979 at depth 8 and 1.000 with no limit. Cross-validation accuracy is 0.620 at depth 8 and 0.595 with no limit. The unlimited tree is perfect on the training fold and about the same as always predicting No on the held-out fold.

## Part 12 Questions

1. Which Decision Tree depth appears to underfit? Depth 1. Training accuracy is 0.661 and cross-validation accuracy is 0.560. Both are poor, and the validation score is below the baseline of 0.605. The tree is too simple to capture the pattern in the study habits.

2. Which depth appears to generalize best? Depth 4. Training accuracy is 0.826 and cross-validation accuracy is 0.670, the highest validation score in the experiment. The tree has learned a pattern that still holds on the held-out students.

3. Do the deepest trees show signs of overfitting? Yes. At depth 8, training accuracy is 0.979 and cross-validation accuracy is 0.620. With no depth limit, training accuracy is 1.000 and cross-validation accuracy is 0.595. Training performance is extremely high, and validation performance is noticeably lower.

4. Why should we not simply choose the tree with the highest training accuracy? That choice is the unlimited tree, because its training accuracy is 1.000. Its cross-validation accuracy is 0.595, below depth 4 and about the same as always predicting No. The highest training score means the tree memorized the training fold. The depth we keep is the one with the best validation accuracy, which is depth 4.

## Part 13 — Select a Decision-Tree Depth

I choose max_depth=4.

Mean cross-validation performance: depth 4's validation accuracy is 0.670. The next depths are 2 at 0.625 and 8 at 0.620. The gap to the closest simpler tree is 0.045, and the gap to depth 8 is 0.050. Depth 1 is 0.560, below the majority baseline of 0.605.

Variability: depth 4 also has the smallest standard deviation, 0.040. Its five folds run from 0.625 to 0.725. Depth 2's standard deviation is 0.057, and the unlimited tree's is 0.060. Depth 8 is nearly as steady, at 0.043, and its mean is 0.050 lower. Depth 4 is the best average and the steadiest.

Model complexity: depth 4 uses fewer layers than depths 6, 8, and the unlimited tree. Those deeper trees have higher training accuracy and lower validation accuracy, so the extra splits are not improving the held-out folds. Depths 1, 2, and 3 are simpler, and they give up too much validation accuracy. A simpler tree would be reasonable if another depth sat in a tight band around 0.670. None does. Depth 2, the closest simpler tree, is 0.045 behind, which is larger than depth 4's own standard deviation of 0.040.

In [ ]:
selected_max_depth = 4

print(depth_experiment_table.to_string(index=False))
print()
print("Selected max_depth:", selected_max_depth)

## Part 14 — Compare the Tuned Tree Again

The tuned tree uses the depth chosen in Part 13. It is scored with the same five folds as the baseline, logistic regression, the original decision tree from Part 8, and the random forest. Precision and recall still treat Yes as the positive class. The test set is not used.

In [ ]:
updated_models = {
    "Majority baseline": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": logistic_model,
    "Decision Tree": tree_model,
    "Tuned Decision Tree": DecisionTreeClassifier(
        max_depth=selected_max_depth, random_state=42
    ),
    "Random Forest": forest_model,
}

updated_rows = []

for name, model in updated_models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", clone(preprocessor)),
        ("classifier", model),
    ])
    scores = cross_validate(
        pipeline, X_train, y_train, cv=cv, scoring=scoring
    )
    updated_rows.append({
        "Model": name,
        "Mean CV Accuracy": scores["test_accuracy"].mean(),
        "Std. Accuracy": scores["test_accuracy"].std(),
        "Precision": scores["test_precision"].mean(),
        "Recall": scores["test_recall"].mean(),
    })

updated_comparison_table = pd.DataFrame(updated_rows).round(3)
chosen_model_name = "Logistic Regression"
print("Model carried forward:", chosen_model_name)
print()
updated_comparison_table

## Part 14 Question

Based only on the training-data cross-validation results, which model would you carry forward to final evaluation? Logistic Regression since mean accuracy is 0.740, its precision is 0.728, and its recall is 0.571. It leads on all three. The random forest is next, at 0.690 accuracy, 0.642 precision, and 0.518 recall, with a larger fold standard deviation of 0.072. The tuned tree uses max_depth=4, the same depth as the original tree, so those two rows match: accuracy 0.670, precision 0.598, and recall 0.493. Tuning confirmed the starting tree. It did not pass logistic regression or the forest. Logistic regression is 0.135 above the baseline, 0.070 above the tuned tree, and 0.050 above the forest. The tree is steadier, with a standard deviation of 0.040, and its mean is lower on every metric. I would carry logistic regression to the final evaluation. This choice uses the 200 training students only.

## 8. Final test evaluation
Choose one workflow, fit on full training data, and evaluate once on the test set.

In [ ]:
# TODO: final test evaluation

from sklearn.metrics import accuracy_score, confusion_matrix

final_model = Pipeline(steps=[
    ("preprocessor", clone(preprocessor)),
    ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
])

final_model.fit(X_train, y_train)
y_pred = final_model.predict(X_test)

test_accuracy = accuracy_score(y_test, y_pred)
test_precision = precision_score(y_test, y_pred, pos_label="Yes", zero_division=0)
test_recall = recall_score(y_test, y_pred, pos_label="Yes", zero_division=0)
test_confusion = confusion_matrix(y_test, y_pred, labels=["No", "Yes"])

baseline_test = DummyClassifier(strategy="most_frequent")
baseline_test.fit(X_train, y_train)
baseline_test_accuracy = accuracy_score(y_test, baseline_test.predict(X_test))

print("Selected model: Logistic Regression")
print("Fit on the 200 training students, evaluated once on the 50 test students")
print()
print("Test accuracy:", round(test_accuracy, 3))
print("Test precision:", round(test_precision, 3))
print("Test recall:", round(test_recall, 3))
print("Majority-class test accuracy:", round(baseline_test_accuracy, 3))
print()
confusion_table = pd.DataFrame(
    test_confusion,
    index=["Actual No", "Actual Yes"],
    columns=["Predicted No", "Predicted Yes"],
)
print("Confusion matrix:")
print(confusion_table)


## Part 15 Questions

1. How does final test accuracy compare with mean CV accuracy? Test accuracy is 0.840. Mean cross-validation accuracy was 0.740. The test score is 0.100 higher.

2. Is it reasonably consistent with what you observed during cross-validation? Yes. The five folds were 0.650, 0.725, 0.725, 0.750, and 0.850, with a standard deviation of 0.064. The test accuracy of 0.840 sits near the highest fold and inside that spread. The test set has only 50 students, so one student changes the accuracy by 0.020. The test result is on the high side of the folds, and it is the same kind of score those folds already produced.

3. Did the model outperform the majority-class baseline? Yes. The baseline test accuracy is 0.600, because 30 of the 50 test students are No and the baseline always predicts No. Logistic regression is right on 42 of the 50, which is 12 more students than the baseline.

4. Are precision and recall similar to the CV results? They are higher than the cross-validation means, and they sit at the high end of the folds. Mean CV precision was 0.728, with folds from 0.600 to 1.000. Test precision is 0.875, inside that range. Mean CV recall was 0.571, with folds from 0.375 to 0.688. Test recall is 0.700, just above the best fold and about one standard deviation above the mean. The test set is a bit kinder than the five-fold average. Precision and recall still say the same thing they said in cross-validation: most Yes predictions are correct, and the model misses some students who actually scored 80 or above. The confusion matrix is 28 true negatives, 2 false positives, 6 false negatives, and 14 true positives.

## 9. Error analysis
Classification: false positives/false negatives. Regression: largest residuals.

In [ ]:
# TODO: error analysis

class_order = list(final_model.named_steps["classifier"].classes_)
yes_column = class_order.index("Yes")

results = X_test.copy()
results.insert(0, "student_id", df.loc[X_test.index, "student_id"])
results["exam_score"] = df.loc[X_test.index, "exam_score"]
results["actual"] = y_test.to_numpy()
results["predicted"] = y_pred
results["yes_probability"] = final_model.predict_proba(X_test)[:, yes_column].round(3)

false_positives = results[
    (results["actual"] == "No") & (results["predicted"] == "Yes")
].sort_values("yes_probability", ascending=False)

false_negatives = results[
    (results["actual"] == "Yes") & (results["predicted"] == "No")
].sort_values("yes_probability", ascending=False)

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))
print()
print("exam_score defines the label. It was not a predictor.")
print()
print("False positives: predicted Yes, actual No")
print(false_positives.to_string(index=False))
print()
print("False negatives: predicted No, actual Yes")
print(false_negatives.to_string(index=False))


## Part 16 Questions

1. How many false positives occurred? 2. 

2. How many false negatives occurred? 6. 

3. Do you notice any patterns among the mistakes? Five of the eight mistakes have a Yes probability between 0.469 and 0.541. The predicted label changes at 0.5, so those five students sit next to the model's cutoff. The two false positives are in that group. S170 has a Yes probability of 0.541, and the habits the model could see include 9.4 study hours and 98.3% attendance. In the training set, the median study hours for Yes students were about 8.9, and the median attendance for Yes students was about 89.5. S133 is similar on those two columns, at 9.7 study hours and 92.9% attendance, with a Yes probability of 0.501. A hypothesis is that the model treated those habits as closer to the training students labeled Yes. That is a hypothesis about the prediction, not a claim that those habits caused either exam score. S133's exam score is 66.8, well below 80, so the habits and the label do not line up for that student. Three false negatives are farther from 0.5. S038, S100, and S090 have Yes probabilities of 0.339, 0.331, and 0.298. Their study hours are 5.5, 5.7, and missing, and their attendance rates are 78.3, 82.7, and 72.7. Those values sit on the low side of the training medians. The model assigned them a clearly lower Yes probability, and the actual label is still Yes.

4. Are some observations likely more difficult to classify than others? Yes. A student whose Yes probability is near 0.5 is a close call for this model. S081, at 0.496, is one of those. A second kind of close call comes from the label itself. high_exam_score flips at 80. S170 scored 79.6 and was predicted Yes. S038 scored 80.1 and was predicted No. The scores are less than half a point apart, and the labels are opposite. The model does not establish why either student received that score. It only shows that a small difference around 80 can put two students in different classes, and that some wrong predictions are much less certain than others.

## Part 17 — Why Not Tune on the Test Set?

One workflow checks the test set after every change:

Try logistic regression, check test accuracy, try a decision tree, check test accuracy, change max_depth, check test accuracy, change max_depth again, check test accuracy, and keep the highest test result.

The workflow we used is different. Cross-validation on the training data compares the models and tunes max_depth. A model is selected from those training-fold scores. The test set is used once, after that selection.

## Part 17 Questions

1. What is wrong with this procedure? The test accuracy is being used to choose the model and the depth. Each check is another look at the same 50 students, and the next try is kept or dropped because of that look. "Choose the highest test result" is model selection done on the test set. The test set was supposed to stay unused until the model and the hyperparameter were already chosen.

2. How has the test set indirectly become part of model development? Those 50 rows are not passed to fit(). Their labels still steer the search. A depth is tried, the test accuracy is read, and another depth is tried when the number is disappointing. The settings that remain are the ones that scored well on those particular students. The test set has taken the role that cross-validation had on the 200 training students.

3. Why is the final test score no longer a completely independent evaluation? An independent evaluation is a score computed after the model is fixed, on students who did not influence the choice. In this procedure the reported score is the best of several scores on the same students. It is the score of the option that won the comparison, so it is expected to sit higher than the score of a model chosen without those looks. In our own folds, logistic regression's accuracy already ran from 0.650 to 0.850. Repeated looks at one group of 50 can land on a high number in the same way one lucky fold did.

The second workflow is preferable because the choices stay on the training folds. We compared the baseline, logistic regression, the decision tree, and the random forest with the same five folds. We chose max_depth from validation accuracy on those folds, and the search selected depth 4. Logistic regression was carried forward because it led on mean accuracy, precision, and recall. Only then was it fit on all 200 training students and scored once on the 50 test students. That test accuracy, 0.840, was not used to pick the model or the depth. It is a check on students who played no role in the selection.

## Part 18 — Scaling and Different Models

The shared pipeline rescales every numeric column with StandardScaler. Logistic regression, the decision tree, and the random forest do not depend on that rescaling in the same way. The next cell checks one tree-style rule on the training students only: attendance_rate < 85, and the same rule after standardization. The test set is not used.

In [ ]:
attendance = X_train["attendance_rate"]
training_mean = attendance.mean()
training_std = attendance.std(ddof=0)
scaled_cutoff = (85 - training_mean) / training_std

raw_below = attendance < 85
scaled_attendance = (attendance - training_mean) / training_std
scaled_below = scaled_attendance < scaled_cutoff

print("Training mean attendance:", round(training_mean, 2))
print("Training standard deviation:", round(training_std, 2))
print("Raw rule: attendance_rate < 85")
print("Scaled rule: scaled attendance <", round(scaled_cutoff, 3))
print("Training students below the raw cutoff:", int(raw_below.sum()))
print("Training students below the scaled cutoff:", int(scaled_below.sum()))
print("Same students:", bool((raw_below == scaled_below).all()))

## Part 18 Questions

1. Why is scaling useful for Logistic Regression? Logistic regression multiplies each feature by a weight and adds those products. The default model also penalizes large weights. Study hours are small numbers, attendance is usually in the 70s to 100, and commute minutes can be near 0 or well over 100. A one-unit step is one hour for study time and one percentage point for attendance, and the penalty treats those weights as if the units were the same. StandardScaler puts each numeric training column on a common scale: 0 at the training mean, and 1 for one training standard deviation. A weight then refers to a comparable step in each column.

2. Why do Decision Trees generally not require scaling? A tree asks a question about one column at a time. A rule such as attendance_rate < 85 uses the order of the students in that column. On the training students, attendance has mean 85.37 and standard deviation 9.63, so the same rule after StandardScaler is scaled attendance < -0.039. The printed threshold changes. The students do not. All 95 training students below 85 are the students below -0.039, and the other training students stay on the other side. Standardizing a column moves the cutoff and leaves the two groups the same. This is different from an algorithm that adds a weight times a feature, or that uses the distance between students. Those calculations change when one column is measured in larger numbers than another.

3. Why do Random Forests generally not require scaling? A random forest is a collection of trees. Each tree splits on the order of values in one feature. Rescaling a column changes the printed cutoff and leaves the groups the same, for the same reason as a single tree. The forest does not add a penalty on the size of a weight, and it does not measure how far two students are in a space where attendance and study hours share an axis. The shared pipeline still scales every model. Logistic regression needs that common scale, and the comparison is easier to read when all three models see the same prepared columns. The scaling does not change a tree's groups the way it changes logistic regression's weights.

## Part 19 — Examine precision/recall instead of accuracy when selecting a tree depth

Accuracy chose max_depth=4. This experiment asks whether precision or recall for the Yes class would choose that same depth. Each depth is scored with the same five folds of the training data. The test set is not used.

My Hypothesis: I expect a deeper tree than depth 4 to have the highest mean cross-validation recall, because accuracy is pulled toward the majority class, No. The depth-4 tree's recall was 0.493, so it missed about half of the held-out Yes students. Extra depth gives the tree more splits that might separate those students on the validation folds. I expect precision not to select that same deeper tree. Deeper trees already overfit on accuracy, and a rule that catches more Yes students can also mark more No students as Yes, which lowers precision.

In [ ]:
metric_depths = [1, 2, 3, 4, 6, 8, None]
metric_rows = []

for depth in metric_depths:
    pipeline = Pipeline(steps=[
        ("preprocessor", clone(preprocessor)),
        ("classifier", DecisionTreeClassifier(max_depth=depth, random_state=42)),
    ])
    scores = cross_validate(
        pipeline, X_train, y_train, cv=cv, scoring=scoring
    )
    metric_rows.append({
        "max_depth": "None" if depth is None else depth,
        "CV Accuracy": scores["test_accuracy"].mean(),
        "CV Precision": scores["test_precision"].mean(),
        "Precision Std.": scores["test_precision"].std(),
        "CV Recall": scores["test_recall"].mean(),
        "Recall Std.": scores["test_recall"].std(),
    })

depth_metric_table = pd.DataFrame(metric_rows).round(3)
print("Depth with highest mean CV accuracy:", depth_metric_table.loc[depth_metric_table["CV Accuracy"].idxmax(), "max_depth"])
print("Depth with highest mean CV precision:", depth_metric_table.loc[depth_metric_table["CV Precision"].idxmax(), "max_depth"])
print("Depth with highest mean CV recall:", depth_metric_table.loc[depth_metric_table["CV Recall"].idxmax(), "max_depth"])
depth_metric_table

## Part 19 Results

| max_depth | CV Accuracy | CV Precision | Precision Std. | CV Recall | Recall Std. |
| --- | --- | --- | --- | --- | --- |
| 1 | 0.560 | 0.349 | 0.187 | 0.312 | 0.185 |
| 2 | 0.625 | 0.574 | 0.127 | 0.341 | 0.114 |
| 3 | 0.610 | 0.521 | 0.074 | 0.509 | 0.183 |
| 4 | 0.670 | 0.598 | 0.046 | 0.493 | 0.132 |
| 6 | 0.615 | 0.512 | 0.081 | 0.471 | 0.145 |
| 8 | 0.620 | 0.522 | 0.057 | 0.520 | 0.079 |
| None | 0.595 | 0.477 | 0.082 | 0.509 | 0.165 |

The results support the hypothesis. Mean recall is highest at depth 8, at 0.520, rather than at depth 4, at 0.493. Mean precision is highest at depth 4, at 0.598. Depth 8's precision is 0.522. Accuracy and precision select depth 4. Recall selects the deeper tree.

The recall gain is small. Depth 8 is 0.027 above depth 4. Depth 3 and the unlimited tree are both at 0.509, only 0.011 behind depth 8. Depth 8's recall standard deviation is 0.079, so that lead over the nearby depths is smaller than the fold-to-fold spread. Selecting on recall alone would pick the more complex tree for a small increase in found Yes students, and that tree is less precise. If the goal is finding students who score 80 or above, depth 8 is the recall choice on these training folds. If the goal is the best balance of correct Yes predictions and overall accuracy, depth 4 remains the choice. Neither choice used the test set.

## 10. Conclusion and AI Reflection
Summarize your evidence. Describe one AI suggestion you rejected or modified and why.

Cross-validation on the 200 training students selected logistic regression. Its mean accuracy was 0.740, its precision was 0.728, and its recall was 0.571, all higher than the baseline, the decision tree, and the random forest. The tree's best validation depth was 4, since the mean accuracy 0.670, the smallest standard deviation in the depth table, and a simpler tree than the depths that drove training accuracy to 1.000. That model was fit once on the full training set. On the 50 test students it scored accuracy 0.840, precision 0.875, and recall 0.700, against a majority-class test accuracy of 0.600. The confusion matrix was 28 true negatives, 2 false positives, 6 false negatives, and 14 true positives.

## Part 20 — AI Reflection

The situation I used AI for was the max_depth experiment. I asked how to compare tree depths without touching the test set, and how to tell underfitting from overfitting. The recommendation was to score depths 1, 2, 3, 4, 6, 8, and unlimited with the same five training folds, and to record mean training accuracy next to mean validation accuracy and its standard deviation. I accepted that design. A deeper tree should fit the training rows more tightly. If validation accuracy rises with it, the extra depth is still useful. If training accuracy keeps rising while validation accuracy falls, the extra depth is memorizing the training fold.

I verified it by running that table. Training accuracy rose from 0.661 at depth 1 to 1.000 with no limit. Validation accuracy peaked at 0.670 at depth 4 and fell to 0.595 with no limit. Depth 1 was below the baseline of 0.605 on the validation folds, which is the underfitting pattern. I modified the first suggested depth list, which also included 5 and 10, down to the seven depths in the assignment table. Depth 4 was still the validation peak, so the shorter list did not change the choice. What I learned is that max_depth is chosen before fit, and the splits inside the tree are what training learns. The plot of the two lines made that difference visible: the training line climbed, and the validation line did not follow it.

The same five folds were the recommendation for the model comparison. I asked for one cross-validation procedure for the baseline, logistic regression, the tree, and the random forest, with accuracy, precision, and recall. The code uses StratifiedKFold with five folds, shuffle, and random_state=42, and the preprocessor is cloned inside each pipeline so each fold refits the medians and the scaler. I checked the baseline against the class counts. The training set has 121 No students out of 200, which is 0.605, and the baseline's mean cross-validation accuracy was 0.605. When precision and recall were added, logistic regression's accuracy stayed 0.740 and the depth-4 tree's stayed 0.670, so the extra metrics did not change the earlier accuracy results.

Did the AI ever suggest using the test set while comparing or tuning models? No. The recommendation was to keep X_test and y_test unused until a model had already been selected from the training folds. The depth search, the model table, and the later precision-and-recall depth experiment all ran on those folds. I followed that. The test set was used once, to score the selected logistic regression, and then to list the eight misclassified test students. Those looks did not choose the model or the depth. In the precision-and-recall experiment, recall preferred depth 8 by 0.027 over depth 4, and I did not go back to the test set to decide which depth was right.